# P9. 분할·학습 파이프라인

기준 원고는 `reports/DS-MINI-Design-울산_4반-박연제.md` 7장·9장이다. 이 노트북은 Batch 1 피처 표만 읽는다. Batch 2·3는 열지 않는다. [D24] [D25]

## 설계서 7장 Modeling Strategy

- 타깃은 `log10(cycle_life)`. MAPE는 예측을 10의 거듭제곱으로 되돌린 사이클 수로 계산한다. 로그는 Batch 2의 이봉과 외삽을 없애지 못한다. [Q1] [Q3]
- 전처리는 `Pipeline(StandardScaler → 모델)`이다. 스케일러는 학습 폴드에서만 fit한다. 1행은 1셀이다.
- Batch 1(36셀, 정책 20종)을 `charging_policy` 그룹 `GroupShuffleSplit`으로 train 80% / hold-out 20%로 나눈다. `RANDOM_STATE=42`. 같은 충전 프로토콜이 train과 hold-out에 동시에 들어가지 않는다. [Q4]
- 튜닝과 Train MAPE는 train 80% 안 `GroupKFold`만 사용한다. Valid는 hold-out 20%(약 7셀)다.
- Batch 2는 모델과 하이퍼파라미터를 잠근 뒤 한 번만 평가한다. Batch 3는 선택 검증이다.
- **메인: ElasticNet, Ridge, Lasso** — ElasticNet은 L1·L2를 함께 써서 소표본에서 계수를 안정적으로 줄인다. 계수가 곧 피처 영향이라 해석이 쉽다. 학습 수명 밖을 외삽할 수 있다 [Q1]. log10(var)와 log 수명이 직선이다 [Q3]. 피처 2개, 학습 셀 36개라 정규화가 과적합을 막는다 [Q5]
- **비교군: 얕은 트리, 부스팅** — 학습 때 본 타깃 범위 밖을 예측하지 못한다. 수명 392인 셀도 534 이상으로만 나온다 [Q1]

딥러닝은 쓰지 않는다. 학습에 쓸 셀이 36개이고, 그 80%만 실제 학습에 들어간다. 파라미터 수가 셀 수를 넘기 쉽다. [Q1] [Q5]

## 이 노트북이 구현하는 D번호

D11, D21, D22, D23, D26, D27, D28, D31, D32, D34, D35, D36. D24·D25는 이번 단계에서 파일을 열지 않는 것으로 지킨다.


In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import DATA_PROCESSED_DIR, RANDOM_STATE
from src.features import MODEL_FEATURES
from src.train import (
    ENET_ALPHAS,
    LASSO_ALPHAS,
    N_SPLITS,
    RIDGE_ALPHAS,
    fit_candidates,
    fold_coefficients,
    holdout_predictions,
    model_coefficients,
    save_final_model,
    select_final,
    split_batch1,
)

# P9는 Batch 1 csv만 연다. [D24] [D25]
path = DATA_PROCESSED_DIR / "features_batch1.csv"
print("읽는 파일", path)
print("모델 입력", MODEL_FEATURES)
print("RANDOM_STATE", RANDOM_STATE, "GroupKFold", N_SPLITS)

train, valid = split_batch1()
n_all = len(train) + len(valid)
overlap = set(train["policy_readable"]) & set(valid["policy_readable"])
print(f"train {len(train)}셀 ({len(train) / n_all * 100:.3f}%), 정책 {train['policy_readable'].nunique()}종")
print(f"hold-out {len(valid)}셀 ({len(valid) / n_all * 100:.3f}%), 정책 {valid['policy_readable'].nunique()}종")
n_policy = train["policy_readable"].nunique() + valid["policy_readable"].nunique()
print(f"정책 비율 {valid['policy_readable'].nunique()}/{n_policy}")
print("겹치는 정책", len(overlap))
print(f"train cycle_life {train['cycle_life'].min():.0f}~{train['cycle_life'].max():.0f}")
print(f"hold-out cycle_life {valid['cycle_life'].min():.0f}~{valid['cycle_life'].max():.0f}")
print("hold-out 정책별 셀 수")
print(valid.groupby("policy_readable").size().to_string())


읽는 파일 data/processed/features_batch1.csv
모델 입력 ('log10_var', 'qd_max_minus_2')
RANDOM_STATE 42 GroupKFold 5
train 29셀 (80.556%), 정책 16종
hold-out 7셀 (19.444%), 정책 4종
정책 비율 4/20
겹치는 정책 0
train cycle_life 534~1054
hold-out cycle_life 617~1074
hold-out 정책별 셀 수
policy_readable
4.4C(80%)-4.4C    1
4.8C(80%)-4.8C    2
7C(40%)-3.6C      2
8C(15%)-3.6C      2


## 분할 결과

train 29셀(36셀 중 80.556%), 정책 16종, cycle_life 534~1,054. hold-out 7셀(19.444%), 정책 4종, cycle_life 617~1,074. 정책 비율은 4/20=20%다. 셀 비율은 20%가 아니다. 정책 그룹 단위라 셀 비율은 근사치다. 겹치는 정책 수는 0이다. [D22]

설계서의 `charging_policy`는 셀 표의 `policy_readable`이다. Batch 1은 newstructure가 없어 두 문자열이 같다.

가이드는 Valid를 셀 단위 hold-out으로 지정한다. 셀마다 충전 프로토콜이 달라, 일반 CV에서는 같은 프로토콜의 셀이 train과 valid로 나뉘는 누수가 생기기 때문이다. 구현은 1행=1셀을 유지한 채 `GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)`의 그룹을 `policy_readable`로 둔다. `test_size`는 셀 비율이 아니라 정책 그룹의 20%다. 같은 프로토콜은 한쪽으로만 간다. 튜닝의 `GroupKFold(n_splits=5)`도 같은 그룹이다. [설계 7장] [D22] [D23]

train에 들어간 수명 최댓값은 1,054다. Batch 1 사용 셀의 최댓값 1,074(b1c5)는 hold-out에 있다.


## 모델 선택 규칙

결과를 보기 전에 잠근 규칙이다. [D23] [D27] [D31]

- 최종 모델은 메인(ElasticNet, Ridge, Lasso) 중에서 고른다. 비교군은 외삽 불가(트리·부스팅)와 피처 수 비교(log10_var 단일 피처)가 목적이라 선택 대상이 아니다.
- GroupKFold 평균 MAPE가 가장 낮은 메인을 고른다. 1위와 차이가 0.5%p 이내인 메인 모델이 있으면, 그 안에서 Valid MAPE가 낮은 쪽을 고른다.
- Valid MAPE 하나만으로 결정하지 않는다.


In [2]:
table = fit_candidates(train, valid)
chosen = select_final(table)

# 이전 잠금(Ridge, alpha=0.001) 대비. 0.01%p 이하이고 모델이 같으면 새 결과로 덮어 쓴다. [D35]
PREV_MODEL = "Ridge"
PREV_CV = 6.656908994528818
PREV_VALID = 4.888609298167768
TOL = 0.01
delta_cv = abs(chosen["cv_mape"] - PREV_CV)
delta_valid = abs(chosen["valid_mape"] - PREV_VALID)
same_lock = chosen["model"] == PREV_MODEL and delta_cv <= TOL and delta_valid <= TOL
print(f"이전 대비 모델 {chosen['model']} dTrain {delta_cv:.6f} dValid {delta_valid:.6f}")
if same_lock:
    model_path = save_final_model(chosen["estimator"])
    print("잠금 덮어씀", model_path)
else:
    model_path = None
    print("판단 필요: final_model.joblib은 덮어쓰지 않음")

show = table.drop(columns=["best_params", "search"]).copy()
show["Train MAPE"] = [
    f"{mean:.3f}±{std:.3f}" for mean, std in zip(show["cv_mape"], show["cv_std"])
]
show["Valid MAPE"] = show["valid_mape"].map(lambda v: f"{v:.3f}")
show = show.rename(columns={"model": "모델", "role": "역할", "params": "최적 하이퍼파라미터"})
print(show[["모델", "역할", "최적 하이퍼파라미터", "Train MAPE", "Valid MAPE"]].to_string(index=False))
print("선택 이유:", chosen["reason"])
print("저장", model_path)
print(
    "격자 끝 alpha",
    f"Ridge {RIDGE_ALPHAS[0]:.4g}~{RIDGE_ALPHAS[-1]:.4g}",
    f"Lasso {LASSO_ALPHAS[0]:.4g}~{LASSO_ALPHAS[-1]:.4g}",
    f"ElasticNet {ENET_ALPHAS[0]:.4g}~{ENET_ALPHAS[-1]:.4g}",
)
grids = {"Ridge": RIDGE_ALPHAS, "Lasso": LASSO_ALPHAS, "ElasticNet": ENET_ALPHAS}
for row in table.itertuples():
    alpha = row.best_params.get("model__alpha")
    if alpha is None:
        continue
    grid = grids[row.model]
    if alpha == grid[0]:
        edge = "하단"
    elif alpha == grid[-1]:
        edge = "상단"
    else:
        edge = "내부"
    print(f"{row.model} alpha {alpha:.4g} 격자 {edge}")


이전 대비 모델 Ridge dTrain 0.000196 dValid 0.000111
잠금 덮어씀 results/final_model.joblib
                         모델  역할                                       최적 하이퍼파라미터   Train MAPE Valid MAPE
                      Ridge  메인                                      alpha=1e-06  6.657±1.217      4.888
                      Lasso  메인                                      alpha=1e-07  6.657±1.217      4.889
                 ElasticNet  메인                        alpha=1e-07, l1_ratio=0.1  6.657±1.217      4.888
               DecisionTree 비교군                                      max_depth=1 10.016±3.912      9.354
                    XGBoost 비교군 learning_rate=0.05, max_depth=2, n_estimators=50  9.261±2.072     10.068
LinearRegression(log10_var) 비교군                                             (없음)  8.874±1.538      8.465
선택 이유: GroupKFold 1위와 0.5%p 이내인 메인이 있어 Valid MAPE가 낮은 쪽을 골랐다. 대상: Ridge CV 6.657% / Valid 4.8885%, ElasticNet CV 6.657% / Valid 4.8885%, Lasso CV 6.657% / Valid 4.8885%. 선택: Ridge.
저장 

In [3]:
pred = holdout_predictions(chosen["estimator"], valid)
view = pred.copy()
view["실제_수명"] = view["실제_수명"].map(lambda v: f"{v:.0f}")
view["예측_수명"] = view["예측_수명"].map(lambda v: f"{v:.3f}")
view["오차%"] = view["오차%"].map(lambda v: f"{v:.3f}")
print("오차% = (예측-실제)/실제×100")
print(view.to_string(index=False))
print(f"hold-out |오차%| 평균 {pred['오차%'].abs().mean():.3f}")


오차% = (예측-실제)/실제×100
cell_id         policy 실제_수명    예측_수명    오차%
  b1c38   7C(40%)-3.6C   617  596.161 -3.377
  b1c39   7C(40%)-3.6C   625  597.809 -4.351
  b1c40   8C(15%)-3.6C   966 1011.863  4.748
  b1c41   8C(15%)-3.6C  1051 1002.202 -4.643
   b1c5 4.4C(80%)-4.4C  1074 1039.676 -3.196
   b1c6 4.8C(80%)-4.8C   636  693.162  8.988
   b1c7 4.8C(80%)-4.8C   870  827.221 -4.917
hold-out |오차%| 평균 4.888


## 선택 규칙 적용 결과

GroupKFold 평균은 세 메인 모두 6.657%다. 1위와 0.5%p 이내라 Valid로 비교했다. 셋째 자리에서 Valid는 Ridge 4.888%, ElasticNet 4.888%, Lasso 4.889%다. 반올림 전 값은 Ridge 4.888498%, ElasticNet 4.888499%, Lasso 4.888505%라 Ridge가 가장 낮다. 최종 모델은 Ridge(alpha=1e-06)다. Valid만으로 고르지 않았고, 비교군은 후보에서 뺐다. [D23] [D31]

세 메인 모델의 차이는 CV 0.000010%p, Valid 0.000007%p로 실질적으로 같다. 규칙에 따라 Ridge를 고른다. Ridge는 계수를 0으로 만들지 않으므로 설계서의 피처 2개를 모두 유지한다. [D41]

비교군 Train MAPE는 단일 피처 8.874%, XGBoost 9.261%, 트리 10.016%다. 메인 세 모델이 단일 피처보다 Train·Valid 모두 낮다. [D27] [D32]

Train MAPE 6.657%(±1.217), Valid MAPE 4.888%. Gap(Train-Valid) = Valid − Train = 4.888 − 6.657 = −1.769%p. hold-out이 7셀이라 Valid가 Train보다 낮은 것을 일반화의 증거로 쓰지 않는다. [D31]

이전 격자(Ridge 하한 0.001) 대비 Train MAPE 차이는 0.000196%p, Valid 차이는 0.000111%p다. 0.01%p를 넘지 않고 모델도 Ridge라 `results/final_model.joblib`을 이 결과로 덮어 잠근다.


In [4]:
import matplotlib.pyplot as plt

from src.config import FIGURES_DIR, set_korean_font
from src.train import main_alpha_curves

set_korean_font()
curves = main_alpha_curves(table)
colors = {"Ridge": "#4C72B0", "Lasso": "#C44E52", "ElasticNet": "#55A868"}
fig, ax = plt.subplots(figsize=(7.2, 4.2))
for name, frame in curves.items():
    label = name if name != "ElasticNet" else "ElasticNet (alpha별 최적 l1_ratio)"
    ax.plot(frame["alpha"], frame["cv_mape"], color=colors[name], label=label)
    best = frame.loc[frame["cv_mape"].idxmin()]
    ax.scatter([best["alpha"]], [best["cv_mape"]], color=colors[name], zorder=3)
    small = frame.loc[frame["alpha"] <= 1e-4, "cv_mape"]
    wide = frame.loc[frame["alpha"] <= 1e-3, "cv_mape"]
    print(
        f"{name} 최적 alpha {best['alpha']:.4g} MAPE {best['cv_mape']:.6f} "
        f"alpha<=1e-4 폭 {float(small.max() - small.min()):.6f} "
        f"alpha<=1e-3 폭 {float(wide.max() - wide.min()):.6f}"
    )
ax.set_xscale("log")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda value, _pos: f"{value:.0e}"))
ax.set_xlabel("alpha")
ax.set_ylabel("GroupKFold 평균 MAPE (%)")
ax.set_title("정규화 강도별 Train MAPE")
ax.legend()
fig.tight_layout()
out = FIGURES_DIR / "modeling_alpha_curve.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.close(fig)
print("저장", out)


Ridge 최적 alpha 1e-06 MAPE 6.656713 alpha<=1e-4 폭 0.000013 alpha<=1e-3 폭 0.000171
Lasso 최적 alpha 1e-07 MAPE 6.656723 alpha<=1e-4 폭 0.010305 alpha<=1e-3 폭 0.102534
ElasticNet 최적 alpha 1e-07 MAPE 6.656714 alpha<=1e-4 폭 0.001457 alpha<=1e-3 폭 0.014528


저장 reports/figures/modeling_alpha_curve.png


## 정규화 강도 해석

<img src="../reports/figures/modeling_alpha_curve.png" width="720">

`reports/figures/modeling_alpha_curve.png`. x축은 alpha(로그), y축은 GroupKFold 평균 MAPE(%). 점은 각 모델의 최적 alpha다. ElasticNet 선은 alpha마다 MAPE가 가장 낮은 l1_ratio다.

1) 목적: 최적 alpha가 이전 격자 하단에 붙어, 더 작은 alpha에서 Train MAPE가 계속 줄어드는지 평탄한지 확인한다. [D35]

2) 결과: 최적 alpha는 0에 가깝고, 작은 alpha 구간은 평탄하다. Ridge 최적은 1e-06, Lasso·ElasticNet 최적은 1e-07(l1_ratio=0.1)로 확장 격자의 하단이다. Ridge는 alpha 1e-06~1e-03에서 MAPE 폭이 0.000171%p다. ElasticNet은 1e-07~1e-04에서 폭 0.001457%p, Lasso는 같은 구간에서 폭 0.010305%p다. alpha가 커지면 MAPE는 15% 근처까지 올라간다.

3) 해석: 피처를 두 개로 잠그고 공선 피처를 배제한 설계(D12, D41, VIF 1.73)가 소표본 과적합 위험을 이미 줄여, 정규화가 추가로 줄일 분산이 적었다.

4) 시사점: 설계서 7장의 메인 선택 이유 중 "외삽 가능", "계수가 피처 영향이라 해석 가능"은 유지되고, "L1·L2 계수 축소"의 실제 효과는 작았다. 정규화 모델을 메인으로 둔 설계는 유지한다. [D26] 하한을 더 내려도 이번 잠금의 MAPE는 0.01%p 이상 좋아지지 않으므로 격자는 여기서 멈춘다.


In [5]:
coef = model_coefficients(chosen["estimator"])
folds = fold_coefficients(chosen["estimator"], train)
print("스케일링 후 계수")
print(coef.map(lambda v: f"{v:.3f}").to_string())
print("GroupKFold 폴드별 계수")
fold_view = folds.copy()
fold_view["log10_var"] = fold_view["log10_var"].map(lambda v: f"{v:.3f}")
fold_view["qd_max_minus_2"] = fold_view["qd_max_minus_2"].map(lambda v: f"{v:.3f}")
print(fold_view.to_string(index=False))
qd_neg = bool((folds["qd_max_minus_2"] < 0).all())
var_neg = bool((folds["log10_var"] < 0).all())
print("log10_var 5폴드 모두 음", var_neg)
print("qd_max_minus_2 5폴드 모두 음", qd_neg)


스케일링 후 계수
log10_var         -0.089
qd_max_minus_2    -0.041
intercept          2.884
GroupKFold 폴드별 계수
 fold  n_train  n_valid_fold log10_var qd_max_minus_2
    1       23             6    -0.094         -0.046
    2       23             6    -0.093         -0.037
    3       23             6    -0.092         -0.044
    4       23             6    -0.089         -0.042
    5       24             5    -0.073         -0.032
log10_var 5폴드 모두 음 True
qd_max_minus_2 5폴드 모두 음 True


## 계수와 D32

스케일링 후 Ridge 계수는 log10(var(ΔQ)) −0.089, qd_max_minus_2 −0.041, 절편 2.884다. 설계 예상은 둘 다 음이다. log10(var)는 수명과 음의 방향이고, qd_max_minus_2는 log10(var)를 통제한 뒤 음이다. 부호가 예상과 같다. [D09] [D10]

GroupKFold 5폴드의 qd_max_minus_2 계수는 −0.046, −0.037, −0.044, −0.042, −0.032로 모두 음이다. log10(var)도 다섯 폴드 모두 음이다. 부호가 폴드마다 유지되므로 D32의 계수 불안정 조건은 해당하지 않는다. 단일 피처 모델은 피처 수 비교용으로 비교군에 남겨 두었고, 메인에서 빼지 않았다. [D32]


## P10으로 넘기는 것

- 잠근 모델: Ridge. `Pipeline(StandardScaler → Ridge)`. 파일 `results/final_model.joblib`. train 80%(29셀)에 재학습한 상태다. hold-out은 학습에 넣지 않았다.
- 하이퍼파라미터: alpha=1e-06, max_iter=10000.
- 피처: log10(var(ΔQ)), qd_max_minus_2. 타깃은 log10(cycle_life). MAPE는 사이클 수.
- Train MAPE(GroupKFold 평균±표준편차): 6.657% ± 1.217
- Valid MAPE(hold-out 7셀): 4.888%
- Batch 2는 이 파일만 불러 한 번 평가한다. 이 노트북에서는 Batch 2·3를 읽지 않았다. [D24] [D36]


# P10. Batch 2 한 번 평가

잠긴 Ridge와 P9 비교군 하이퍼파라미터를 그대로 쓴다. 이 결과를 보고 모델, 하이퍼파라미터, 피처, 분할은 바꾸지 않는다. [D24] [D36]

Batch 2 파일은 바로 다음 코드 셀이 아니라, 그 다음의 예측 셀에서 `load_features(2)`로 한 번만 읽는다.


In [6]:
from src.config import RESULTS_DIR
from src.train import fit_locked_comparisons, load_final_model, split_batch1

ridge = load_final_model()
train_locked, _valid_locked = split_batch1()
comparisons = fit_locked_comparisons(train_locked)
loaded_path = RESULTS_DIR / "final_model.joblib"
ridge_step = ridge.named_steps["model"]
print("파일", loaded_path)
print("Ridge", type(ridge_step).__name__, f"alpha={ridge_step.alpha:.6g}", f"max_iter={ridge_step.max_iter}")
print("Ridge 학습 셀 수", int(ridge.named_steps["scaler"].n_samples_seen_))
print("비교군 재학습 셀 수", len(train_locked))
print(f"train cycle_life {train_locked['cycle_life'].min():.0f}~{train_locked['cycle_life'].max():.0f}")
for column in ("log10_var", "qd_max_minus_2"):
    print(
        f"train {column} {train_locked[column].min():.6f}~{train_locked[column].max():.6f}"
    )
for name, spec in comparisons.items():
    seen = int(spec["pipe"].named_steps["scaler"].n_samples_seen_)
    print(name, spec["params"], "학습 셀", seen)
if len(train_locked) != 29 or int(ridge.named_steps["scaler"].n_samples_seen_) != 29:
    raise RuntimeError("학습 셀 수가 29가 아니다. Batch 2를 읽기 전에 멈춘다.")
print("평가 전 고정 확인: 모델 4개, 학습 셀 29. Batch 2는 아직 읽지 않음.")


파일 results/final_model.joblib
Ridge Ridge alpha=1e-06 max_iter=10000
Ridge 학습 셀 수 29
비교군 재학습 셀 수 29
train cycle_life 534~1054
train log10_var -4.146462~-3.349904
train qd_max_minus_2 0.003094~0.009460
DecisionTree max_depth=1, min_samples_leaf=3 학습 셀 29
XGBoost max_depth=2, n_estimators=50, learning_rate=0.05 학습 셀 29
LinearRegression(log10_var) 피처 log10_var 하나 학습 셀 29
평가 전 고정 확인: 모델 4개, 학습 셀 29. Batch 2는 아직 읽지 않음.


In [7]:
import numpy as np

from src.config import RESULTS_DIR
from src.train import (
    build_batch2_predictions,
    feature_range_table,
    life_band_table,
    load_features,
    mape_original,
    performance_table,
)

# Batch 2는 이 셀에서만 읽는다. 예측도 이 셀에서 한 번. [D24]
batch2 = load_features(2)
print("읽는 파일 data/processed/features_batch2.csv", "행", len(batch2), "batch", sorted(batch2["batch"].unique()))
pred = build_batch2_predictions(batch2, train_locked, ridge, comparisons)
aligned = batch2.set_index("cell_id").loc[pred["cell_id"]]
test_mape = mape_original(aligned["log_life"], np.log10(pred["Ridge_예측"].to_numpy()))
print(f"Test MAPE 원값 {test_mape:.6f}")

report = performance_table(test_mape)
report_path = RESULTS_DIR / "model_performance.csv"
report.to_csv(report_path, index=False)
print(report.to_string(index=False))
print("저장", report_path)

pred_path = RESULTS_DIR / "predictions_batch2.csv"
pred.to_csv(pred_path, index=False)
print("저장", pred_path, "행", len(pred))

show = pred.copy()
for column in ("cycle_life", "Ridge_예측", "DecisionTree_예측", "XGBoost_예측", "log10_var_단일_예측"):
    show[column] = show[column].map(lambda value: f"{value:.3f}")
show["오차%"] = pred["오차%"].map(lambda value: f"{value:.3f}")
show["|오차%|"] = pred["|오차%|"].map(lambda value: f"{value:.3f}")
show["log10_var"] = pred["log10_var"].map(lambda value: f"{value:.3f}")
show["qd_max_minus_2"] = pred["qd_max_minus_2"].map(lambda value: f"{value:.6f}")
print(show.to_string(index=False))

bands = life_band_table(pred)
band_show = bands.copy()
for column in band_show.columns:
    if column in ("수명_구간", "셀_수", "과대예측_셀_수"):
        continue
    band_show[column] = bands[column].map(lambda value: f"{value:.3f}")
print("수명 구간")
print(band_show.to_string(index=False))

print(
    "실제 최단 수명", f"{pred['cycle_life'].min():.0f}",
    "DecisionTree 예측 최소", f"{pred['DecisionTree_예측'].min():.3f}",
    "XGBoost 예측 최소", f"{pred['XGBoost_예측'].min():.3f}",
    "Ridge 예측 최소", f"{pred['Ridge_예측'].min():.3f}",
)
print(
    "train 수명 최소", f"{train_locked['cycle_life'].min():.0f}",
    "DecisionTree가 train 최소보다 작은 셀", int((pred["DecisionTree_예측"] < train_locked["cycle_life"].min()).sum()),
    "XGBoost가 train 최소보다 작은 셀", int((pred["XGBoost_예측"] < train_locked["cycle_life"].min()).sum()),
)

ranges = feature_range_table(pred)
range_show = ranges.copy()
range_show["Ridge_MAPE"] = ranges["Ridge_MAPE"].map(lambda value: f"{value:.3f}" if value == value else "")
print("피처 train 범위")
print(range_show.to_string(index=False))
either = (pred["log10_var_범위"] != "안") | (pred["qd_max_minus_2_범위"] != "안")
outside = pred.loc[either]
print("두 피처 중 하나라도 범위 밖", int(either.sum()), "Ridge MAPE", f"{outside['|오차%|'].mean():.3f}" if len(outside) else "없음")


읽는 파일 data/processed/features_batch2.csv 행 39 batch [np.int64(2)]


Test MAPE 원값 20.258837
                      구분  MAPE (%)                                  비고
      Train (Batch 1 CV)     6.657 29 / 7 / 39. GroupKFold 평균. 학습 셀 29
Valid (Batch 1 Hold-out)     4.888           29 / 7 / 39. hold-out 셀 7
          Test (Batch 2)    20.259               29 / 7 / 39. 테스트 셀 39
       Gap (Train-Valid)    -1.769           29 / 7 / 39. (+) : 과적합 의심
        Gap (Valid-Test)    15.371    29 / 7 / 39. (+) : 배치간 일반화 저하 의심
       Gap (Target-Test)    11.159      29 / 7 / 39. Target : 원논문 9.1%
저장 results/model_performance.csv
저장 results/predictions_batch2.csv 행 39
cell_id             policy_readable cycle_life     수명_구간 log10_var qd_max_minus_2 log10_var_범위 qd_max_minus_2_범위 Ridge_예측     오차%  |오차%| DecisionTree_예측 XGBoost_예측 log10_var_단일_예측
   b2c0                5.2C(58%)-4C    477.000      <534    -3.487       0.008114            안                 안  476.637  -0.076  0.076         666.987    715.128         636.424
   b2c1              5.6C(26%)-4.5C    491.000  

## Batch 2 해석

1) 목적: 잠긴 Ridge의 Test MAPE와 세 Gap이 설계 8장의 외삽, 과대예측, 원논문 비교 한계와 맞는지를 한 번의 평가로 확인한다. [D24] [D30] [D33]

2) 결과: Train 6.657%(셀 29), Valid 4.888%(셀 7), Test 20.259%(셀 39). Gap(Train-Valid)=−1.769, Gap(Valid-Test)=15.371, Gap(Target-Test)=11.159. 수명 구간은 <534가 30셀, 534~1,074가 7셀, >1,074가 2셀이다. Ridge MAPE는 15.121 / 37.701 / 36.277, 평균 부호 오차는 +1.124 / +31.999 / +36.277%p, 과대예측은 15 / 6 / 2셀이다. DecisionTree 예측 최솟값은 666.987, XGBoost는 588.958이다. 둘 다 train 수명 최소 534보다 작지 않다. 실제 최단 수명은 392이고 Ridge 예측 최솟값은 259.357이다. log10_var가 train 범위 밖인 셀은 아래 7, 위 11이고 Ridge MAPE는 40.813 / 16.865이다. qd_max_minus_2는 아래 9, 위 12, MAPE 35.071 / 17.930이다. 두 피처 중 하나라도 밖인 셀은 28개, Ridge MAPE 23.423이다.

3) 해석: Gap(Train-Valid)가 음수라 과적합 부호는 아니다. hold-out이 7셀이라 이 음수를 Batch 1 안에서의 일반화 성공으로 쓰지 않는다. [D31] Gap(Valid-Test)는 양수다. 설계 8장대로 과대예측을 먼저 봤다. 534 이상 9셀은 평균 부호가 +32.0%p, +36.3%p로 과대예측이다. 534 미만 30셀은 평균 부호 +1.124%p이고 과대예측이 15셀이라 부호가 상쇄된다. 배치 번호를 피처로 넣어 맞추지 않는다. [D30] Gap(Target-Test)는 양수다. 원논문 9.1%는 학습·테스트 구성과 피처 수가 달라 직접 비교에 한계가 있고, 차이는 배치 간 분포 차이로 해석한다. [D33] [설계 8장]

4) 시사점: 모델, 하이퍼파라미터, 피처, 분할은 유지한다. 트리는 수명 392를 534 이상으로만 내므로 외삽 구간의 비교군으로 남긴다. [D27] [D29]

## 설계 예상 vs 결과

| 예상 | 결과 | 맞았는가 |
| --- | --- | --- |
| Batch 2의 30/39는 534 미만, 1,074 초과는 2셀. 오차를 그 구간으로 나눈다 [D29] | <534 30셀, 534~1,074 7셀, >1,074 2셀. Ridge MAPE 15.121 / 37.701 / 36.277 | 맞음 |
| 트리는 수명 392도 534 이상으로만 예측한다 [D27] | DecisionTree 최소 666.987, XGBoost 최소 588.958. train 최소보다 작은 예측 0. Ridge 최소 259.357 | 맞음 |
| Gap(Valid-Test)가 양수면 과대예측을 먼저 의심한다. 배치 번호로 맞추지 않는다 [D30] | Gap 15.371. <534 평균 부호 +1.124%(15/30), 534~1,074 +31.999%(6/7), >1,074 +36.277%(2/2). 배치 번호는 피처에 없다 | 양수는 맞음. 과대예측은 534 이상에서 맞고, 534 미만은 상쇄 |
| Valid는 약 7셀이라 한 장의 MAPE로 확정하지 않는다 [D31] | Gap(Train-Valid)=−1.769. 과적합 부호가 아니다 | 맞음. 음수를 일반화의 증거로 쓰지 않음 |
| Gap(Target-Test)가 양수일 가능성이 높다. 원논문 9.1%는 구성이 달라 직접 비교에 한계가 있고, 원인은 배치 간 분포 차이다 [D33] | Test 20.259, Gap 11.159. 피처 2개, 학습은 Batch 1뿐 | 맞음 |


# P11. 오류 분석

`results/predictions_batch2.csv`와 P10에서 고정한 Ridge·비교군만 쓴다. 모델, 피처, 분할은 바꾸지 않고 Batch 2 수명은 다시 예측하지 않는다. [D24] [D38]

`features_batch2.csv`는 P10 예측 셀의 `load_features(2)` 한 번이 전부다. 3.0V ΔQ만 `delta_q`를 위해 pkl의 Qdlin을 읽는다.


In [8]:
import numpy as np
import pandas as pd

from src.config import RESULTS_DIR
from src.features import MODEL_FEATURES

recorded = pd.read_csv(RESULTS_DIR / "predictions_batch2.csv")
scaler = ridge.named_steps["scaler"]
linear = ridge.named_steps["model"]
coef = np.asarray(linear.coef_, dtype=float).ravel()
intercept = float(np.asarray(linear.intercept_).reshape(-1)[0])
z_train = scaler.transform(train_locked.loc[:, list(MODEL_FEATURES)])
z_test = scaler.transform(recorded.loc[:, list(MODEL_FEATURES)])
c_train = z_train * coef
c_test = z_test * coef
recon = intercept + c_test.sum(axis=1)
gap = np.max(np.abs(recon - np.log10(recorded["Ridge_예측"].to_numpy(float))))
print("계수", dict(zip(MODEL_FEATURES, np.round(coef, 6))), "절편", round(intercept, 6))
print("log10 예측 재구성 최대 차이", f"{gap:.3e}")
print(
    "train 29 기여 평균",
    f"log10_var {c_train[:, 0].mean():.6f}",
    f"qd_max_minus_2 {c_train[:, 1].mean():.6f}",
)

rows = []
for band in ("<534", "534~1,074", ">1,074"):
    mask = (recorded["수명_구간"] == band).to_numpy()
    part = recorded.loc[mask]
    single = (part["log10_var_단일_예측"] - part["cycle_life"]) / part["cycle_life"] * 100.0
    rows.append({
        "수명_구간": band,
        "셀_수": int(mask.sum()),
        "log10_var_기여": float(c_test[mask, 0].mean()),
        "train_대비_var": float(c_test[mask, 0].mean() - c_train[:, 0].mean()),
        "qd_기여": float(c_test[mask, 1].mean()),
        "train_대비_qd": float(c_test[mask, 1].mean() - c_train[:, 1].mean()),
        "Ridge_평균_부호_오차%": float(part["오차%"].mean()),
        "단일_평균_부호_오차%": float(single.mean()),
    })
contrib_table = pd.DataFrame(rows)
show = contrib_table.copy()
for column in show.columns:
    if column in ("수명_구간", "셀_수"):
        continue
    show[column] = contrib_table[column].map(lambda value: f"{value:.3f}")
print(show.to_string(index=False))


계수 {'log10_var': np.float64(-0.088708), 'qd_max_minus_2': np.float64(-0.040537)} 절편 2.88435
log10 예측 재구성 최대 차이 4.441e-16
train 29 기여 평균 log10_var 0.000000 qd_max_minus_2 0.000000
    수명_구간  셀_수 log10_var_기여 train_대비_var  qd_기여 train_대비_qd Ridge_평균_부호_오차% 단일_평균_부호_오차%
     <534   30       -0.155       -0.155 -0.078      -0.078           1.124       33.865
534~1,074    7        0.183        0.183 -0.017      -0.017          31.999       17.623
   >1,074    2        0.227        0.227  0.087       0.087          36.277       -6.060


In [9]:
import pickle

from src.config import DATA_PROCESSED_DIR
from src.features import delta_q

# 예측은 하지 않는다. ΔQ 곡선만 읽는다. [D08] [D38]
dq_rows = []
median_value = {}
for batch in (1, 2):
    with (DATA_PROCESSED_DIR / f"batch{batch}.pkl").open("rb") as handle:
        payload = pickle.load(handle)
    cells = payload["cells"]
    cells = cells.loc[cells["use"]].reset_index(drop=True)
    vdlin = np.asarray(payload["vdlin"], dtype=float)
    voltage_index = int(np.argmin(np.abs(vdlin - 3.0)))
    band_curves = []
    for record in cells.itertuples(index=False):
        curve = delta_q(payload["qdlin"][record.cell_id])
        point = float(curve[voltage_index])
        dq_rows.append({
            "cell_id": record.cell_id,
            "batch": batch,
            "cycle_life": float(record.cycle_life),
            "dq_3v": point,
        })
        if 700 <= float(record.cycle_life) <= 1100:
            band_curves.append(curve)
    median_curve = np.median(np.vstack(band_curves), axis=0)
    median_value[batch] = (len(band_curves), float(median_curve[voltage_index]))

dq_table = pd.DataFrame(dq_rows)
design_dq = {1: -0.031, 2: -0.021}
print("3.0V 인덱스", voltage_index, "전압", f"{vdlin[voltage_index]:.4f}")
repro_rows = []
for batch, design in design_dq.items():
    count, value = median_value[batch]
    rounded = float(f"{value:.3f}")
    repro_rows.append({
        "배치": batch,
        "셀_수": count,
        "설계서_값": f"{design:.3f}",
        "재현_값": f"{value:.4f}",
        "소수_셋째": f"{rounded:.3f}",
        "일치": "일치" if rounded == design else "불일치",
    })
repro = pd.DataFrame(repro_rows)
print(repro.to_string(index=False))

recorded = recorded.merge(dq_table.loc[dq_table["batch"] == 2, ["cell_id", "dq_3v"]], on="cell_id", how="left")
if recorded["dq_3v"].isna().any():
    raise RuntimeError("Batch 2 예측 셀에 3.0V ΔQ가 없다.")
recorded["깊이_분위"] = pd.qcut(recorded["dq_3v"], 3, labels=["깊음", "중간", "얕음"])
depth_rows = []
for label, part in recorded.groupby("깊이_분위", observed=True):
    depth_rows.append({
        "깊이_분위": str(label),
        "셀_수": int(len(part)),
        "dq_3v_최소": float(part["dq_3v"].min()),
        "dq_3v_최대": float(part["dq_3v"].max()),
        "Ridge_MAPE": float(part["|오차%|"].mean()),
        "Ridge_평균_부호_오차%": float(part["오차%"].mean()),
    })
depth_table = pd.DataFrame(depth_rows)
depth_show = depth_table.copy()
for column in ("dq_3v_최소", "dq_3v_최대", "Ridge_MAPE", "Ridge_평균_부호_오차%"):
    depth_show[column] = depth_table[column].map(lambda value: f"{value:.3f}")
print(depth_show.to_string(index=False))

train_policies = set(train_locked["policy_readable"])
top = recorded.nlargest(5, "|오차%|").copy()
top["정책_train에_있음"] = top["policy_readable"].isin(train_policies)
top_show = pd.DataFrame({
    "cell_id": top["cell_id"],
    "정책": top["policy_readable"],
    "실제_수명": top["cycle_life"].map(lambda value: f"{value:.0f}"),
    "예측_수명": top["Ridge_예측"].map(lambda value: f"{value:.3f}"),
    "오차%": top["오차%"].map(lambda value: f"{value:.3f}"),
    "수명_구간": top["수명_구간"],
    "log10_var_범위": top["log10_var_범위"],
    "qd_범위": top["qd_max_minus_2_범위"],
    "dq_3v": top["dq_3v"].map(lambda value: f"{value:.4f}"),
    "정책_train에_있음": top["정책_train에_있음"],
})
print(top_show.to_string(index=False))


3.0V 인덱스 333 전압 3.0000
 배치  셀_수  설계서_값    재현_값  소수_셋째 일치
  1   26 -0.031 -0.0312 -0.031 일치
  2    7 -0.021 -0.0209 -0.021 일치
깊이_분위  셀_수 dq_3v_최소 dq_3v_최대 Ridge_MAPE Ridge_평균_부호_오차%
   깊음   13   -0.084   -0.059     16.285          11.312
   중간   13   -0.054   -0.048     15.960          -6.133
   얕음   13   -0.048   -0.006     28.531          20.228
cell_id                          정책 실제_수명    예측_수명     오차%     수명_구간 log10_var_범위 qd_범위   dq_3v  정책_train에_있음
  b2c44   5.2C(58%)-4C-newstructure   841 1506.860  79.175 534~1,074           아래    아래 -0.0185         False
   b2c9 5.6C(26%)-4.5C-newstructure   791 1286.881  62.690 534~1,074           아래     안 -0.0125         False
  b2c34 5.6C(26%)-4.5C-newstructure  1186 1732.842  46.108    >1,074           아래    아래 -0.0169         False
  b2c17              5.6C(26%)-4.5C   471  259.357 -44.935      <534            안     위 -0.0522         False
   b2c4               4.65C(69%)-6C   444  585.921  31.964      <534            위    아래 -0.0683      

In [10]:
import matplotlib.pyplot as plt

from src.config import FIGURES_DIR, set_korean_font

set_korean_font()
band_colors = {"<534": "#C44E52", "534~1,074": "#4C72B0", ">1,074": "#55A868"}
fig, ax = plt.subplots(figsize=(6.2, 5.4))
ax.axvspan(534, 1054, color="#D9D9D9", zorder=0)
ax.axhspan(534, 1054, color="#D9D9D9", zorder=0)
ax.scatter(
    recorded["cycle_life"],
    recorded["DecisionTree_예측"],
    s=16,
    c="0.45",
    label="DecisionTree",
    zorder=2,
)
for band, color in band_colors.items():
    part = recorded.loc[recorded["수명_구간"] == band]
    ax.scatter(
        part["cycle_life"],
        part["Ridge_예측"],
        s=32,
        c=color,
        label=f"Ridge {band}",
        zorder=3,
    )
limits = (280, 2200)
ax.plot(limits, limits, color="0.15", lw=1, label="y=x", zorder=1)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(*limits)
ax.set_ylim(*limits)
ax.set_xlabel("실제 수명 (사이클)")
ax.set_ylabel("예측 수명 (사이클)")
ax.set_title("실제 수명과 예측")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "err_actual_vs_pred.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("DecisionTree 예측 최소", f"{recorded['DecisionTree_예측'].min():.3f}")
print("저장", FIGURES_DIR / "err_actual_vs_pred.png")

mean_var = float(scaler.mean_[0])
scale_var = float(scaler.scale_[0])
xs = np.linspace(
    min(train_locked["log10_var"].min(), recorded["log10_var"].min()),
    max(train_locked["log10_var"].max(), recorded["log10_var"].max()),
    80,
)
line = intercept + coef[0] * ((xs - mean_var) / scale_var)
fig, ax = plt.subplots(figsize=(6.4, 5.0))
ax.scatter(
    train_locked["log10_var"],
    np.log10(train_locked["cycle_life"]),
    s=28,
    c="#4C72B0",
    label="Batch 1 train 29",
    zorder=3,
)
ax.scatter(
    recorded["log10_var"],
    np.log10(recorded["cycle_life"]),
    s=28,
    c="#C44E52",
    label="Batch 2 39",
    zorder=2,
)
ax.plot(xs, line, color="0.15", lw=1.4, label="Ridge (qd=train 평균)", zorder=1)
ax.set_xlabel("log10(var(ΔQ))")
ax.set_ylabel("log10(cycle_life)")
ax.set_title("같은 분산 축에서 본 수명")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "err_offset.png", dpi=150, bbox_inches="tight")
plt.close(fig)
overlap = recorded.loc[recorded["수명_구간"] == "534~1,074", "log10_var"]
print(
    "log10_var 중앙 train",
    f"{train_locked['log10_var'].median():.3f}",
    "Batch2 534~1,074",
    f"{overlap.median():.3f}",
    "Batch2 전체",
    f"{recorded['log10_var'].median():.3f}",
)
print("저장", FIGURES_DIR / "err_offset.png")


DecisionTree 예측 최소 666.987
저장 reports/figures/err_actual_vs_pred.png
log10_var 중앙 train -3.777 Batch2 534~1,074 -4.231 Batch2 전체 -3.487
저장 reports/figures/err_offset.png


## 기여 분해와 D30 판정

log10 예측 = 절편 2.884 + 계수 × 표준화 값. 스케일러는 train 29셀에 고정된 것이다. train 29셀의 두 기여 평균은 0.000이다. 재구성과 저장된 Ridge 예측의 log10 차이는 4.441e-16이다.

| 수명 구간 | 셀 수 | log10_var 기여 | train 대비 | qd_max_minus_2 기여 | train 대비 | Ridge 평균 부호 % | log10_var 단일 평균 부호 % |
| --- | --- | --- | --- | --- | --- | --- | --- |
| <534 | 30 | −0.155 | −0.155 | −0.078 | −0.078 | +1.124 | +33.865 |
| 534~1,074 | 7 | +0.183 | +0.183 | −0.017 | −0.017 | +31.999 | +17.623 |
| >1,074 | 2 | +0.227 | +0.227 | +0.087 | +0.087 | +36.277 | −6.060 |

판정: **맞다.**

534 미만에서 과대예측이 상쇄된 원인은 `qd_max_minus_2` 기여다. 이 구간 Ridge 평균 부호 오차는 +1.124%인데, log10_var만 있는 모델은 +33.865%다. 이 값은 같은 구간 MAPE와 같아서 과소예측이 섞여 상쇄된 것이 아니다. `qd_max_minus_2`가 없을 때 과대예측이 더 크다.

기여 평균은 log10_var −0.155, `qd_max_minus_2` −0.078이다. 계수가 둘 다 음이라, 짧은 셀은 두 피처가 train보다 커서 예측 수명을 줄인다. 분산 기여의 절대값이 더 크지만, 분산만으로는 과대예측이 +33.865%로 남는다. 상쇄를 +1.124%까지 내리는 쪽은 `qd_max_minus_2`다.

534 이상은 다르다. 534~1,074은 분산 기여 +0.183이 수명을 늘리고 `qd_max_minus_2`는 −0.017에 그쳐 평균 부호 +31.999%다. 1,074 초과는 두 기여가 모두 양이고 평균 부호 +36.277%다. 단일 모델은 이 2셀에서 −6.060%라, 긴 쪽의 과대예측은 `qd_max_minus_2`가 더한다. [D30]


## 3.0V ΔQ 깊이

수명 700~1,100 셀의 ΔQ 중앙 곡선을 만든 뒤 3.0V 한 점을 읽었다. `delta_q`는 기존 함수다. [D08] [D38]

| 배치 | 셀 수 | 설계서 값 (Ah) | 재현 값 (Ah) | 소수 셋째 | 일치 |
| --- | --- | --- | --- | --- | --- |
| Batch 1 | 26 | −0.031 | −0.0312 | −0.031 | 일치 |
| Batch 2 | 7 | −0.021 | −0.0209 | −0.021 | 일치 |

Batch 2 39셀을 셀별 3.0V ΔQ의 3분위로 나눴다. 깊음이 더 음수다.

| 깊이 | 셀 수 | 3.0V ΔQ (Ah) | Ridge MAPE | 평균 부호 % |
| --- | --- | --- | --- | --- |
| 깊음 | 13 | −0.084 ~ −0.059 | 16.285 | +11.312 |
| 중간 | 13 | −0.054 ~ −0.048 | 15.960 | −6.133 |
| 얕음 | 13 | −0.048 ~ −0.006 | 28.531 | +20.228 |

얕은 13셀의 MAPE와 과대예측이 가장 크다. 같은 수명인데 Batch 2의 골이 얕으면 수명을 길게 보는 예상과 맞다. [D30] [D38]


## 오차 상위 5셀

| cell_id | 정책 | 실제 | Ridge 예측 | 오차% | 수명 구간 | log10_var | qd_max_minus_2 | 3.0V ΔQ | train 정책 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| b2c44 | 5.2C(58%)-4C-newstructure | 841 | 1506.860 | +79.175 | 534~1,074 | 아래 | 아래 | −0.0185 | 없음 |
| b2c9 | 5.6C(26%)-4.5C-newstructure | 791 | 1286.881 | +62.690 | 534~1,074 | 아래 | 안 | −0.0125 | 없음 |
| b2c34 | 5.6C(26%)-4.5C-newstructure | 1,186 | 1732.842 | +46.108 | >1,074 | 아래 | 아래 | −0.0169 | 없음 |
| b2c17 | 5.6C(26%)-4.5C | 471 | 259.357 | −44.935 | <534 | 안 | 위 | −0.0522 | 없음 |
| b2c4 | 4.65C(69%)-6C | 444 | 585.921 | +31.964 | <534 | 위 | 아래 | −0.0683 | 없음 |

상위 5셀의 정책은 Batch 1 train 16종에 없다. 오차가 큰 세 셀(b2c44, b2c9, b2c34)은 newstructure이고, log10_var가 train 범위보다 아래이며 3.0V ΔQ가 −0.019~−0.013Ah로 얕다. 예측이 실제보다 46~79% 길다. 나머지 둘은 수명 534 미만이라 방향이 다르고, b2c17은 qd_max_minus_2가 train보다 커서 −44.935%다.


## 그림

<img src="../reports/figures/err_actual_vs_pred.png" width="640">

`reports/figures/err_actual_vs_pred.png`. x축은 실제 수명, y축은 예측 수명. 둘 다 로그. 직선은 y=x, 음영은 train 수명 534~1,054.

1. 목적: Batch 2에서 예측이 어느 수명에서 벗어나는지, DecisionTree가 534 아래로 내려가지 못하는지를 본다. [D27]
2. 결과: Ridge의 534 미만 30셀은 y=x 주변에 있고 평균 부호 오차 +1.124%다. 534~1,074 7셀은 평균 부호 +31.999%, 1,074 초과 2셀은 +36.277%로 직선 위에 있다. DecisionTree 예측 최소는 666.987이라, 실제 392~514인 셀도 그 높이에 가로로 모인다.
3. 해석: 트리는 학습한 수명 아래로 외삽하지 못한다. Ridge는 259.357까지 내려가 짧은 쪽은 맞추지만, ΔQ 분산이 train보다 작은 긴 셀은 수명을 늘려 예측한다. [D27] [D30]
4. 시사점: 트리는 외삽 실패를 보여주는 비교군으로 둔다. 오차의 본류는 534 이상 과대예측이고, 이 그림 때문에 모델을 바꾸지 않는다.

<img src="../reports/figures/err_offset.png" width="640">

`reports/figures/err_offset.png`. x축은 log10(var(ΔQ)), y축은 log10(cycle_life). 검은 선은 qd_max_minus_2를 train 평균에 고정한 Ridge다.

1. 목적: 같은 수명에서 Batch 2의 분산이 Batch 1 train보다 왼쪽인지 본다. [D30]
2. 결과: train 29셀 log10_var 중앙은 −3.777이고, 수명 534~1,074인 Batch 2 7셀 중앙은 −4.231이다. 같은 수명대에서 Batch 2가 왼쪽에 있다. 534 미만 30셀은 −3.5 근처에 모여 선 아래·옆에 있다. Batch 2 전체 중앙은 −3.487이다.
3. 해석: 계수가 음이라 분산이 작을수록 선은 수명을 길게 본다. 534 이상 Batch 2는 수명이 비슷한데 분산이 더 작아, 이 선이 수명을 과대예측한다. 짧은 셀은 분산이 커서 선이 수명을 줄인다.
4. 시사점: 배치 간 ΔQ 오프셋은 534 이상에서 보인다. 534 미만의 상쇄와 섞지 않는다. 배치 번호는 피처로 넣지 않는다. [D30]


## 원인 가설과 개선 방향

구현하지 않는다.

1. 학습 배치가 하나라 배치 간 ΔQ 오프셋을 배울 수 없다. 여러 배치를 섞어 학습하는 원논문 방식이 이 차이를 다루는 길이다. [D30] [D33]
2. 정규화 ΔQ는 DAY 1에서 오프셋을 줄이지 못했다. 3.0V 깊이 비는 1.30에서 1.29로만 바뀌었다. 배치에 덜 민감한 다른 피처를 찾을 여지가 있다. [D17]
3. 새 배치가 들어오면 초기 소수 셀로 절편만 다시 맞추는 운영을 둘 수 있다. 이것은 배치 번호를 피처로 넣는 것과 다르다. [D30]


## 오류 분석 요약

Test MAPE 20.259는 534 이상 9셀의 과대예측(평균 부호 +31.999%, +36.277%)에서 커졌고, 534 미만 30셀은 +1.124%로 상쇄됐다.
534 미만 상쇄는 qd_max_minus_2 기여다. 기여 평균 −0.078이고, log10_var만 쓰면 평균 부호 +33.865%로 MAPE와 같다.
수명 700~1,100의 3.0V 중앙 ΔQ는 Batch 1 −0.031, Batch 2 −0.021Ah로 재현됐다. ΔQ가 얕은 13셀의 Ridge MAPE는 28.531이다.
오차가 큰 셀은 newstructure 장수명(b2c44, b2c9, b2c34)으로, log10_var가 train보다 작고 그 정책은 train에 없다.
개선은 구현하지 않았다. 후보는 배치 혼합 학습, 배치에 덜 민감한 피처, 새 배치의 절편 재보정이다.


# P12. Batch 3 한 번 평가

P10에서 고정한 Ridge와 비교군 3개를 그대로 쓴다. 모델, 하이퍼파라미터, 피처, 분할은 결과를 본 뒤에도 바꾸지 않는다. [D25] [D39]

`features_batch3.csv`는 아래 예측 셀의 `load_features(3)` 한 번이 전부다. Batch 2 예측은 저장된 `predictions_batch2.csv`만 읽는다.


In [11]:
loaded_again = load_final_model()
ridge_step = loaded_again.named_steps["model"]
print("파일", RESULTS_DIR / "final_model.joblib")
print("Ridge", type(ridge_step).__name__, f"alpha={ridge_step.alpha:.6g}", f"max_iter={ridge_step.max_iter}")
print("Ridge 학습 셀 수", int(loaded_again.named_steps["scaler"].n_samples_seen_))
print("비교군은 P10 적합 유지, 재튜닝 없음. 학습 셀", len(train_locked))
print(f"train cycle_life {train_locked['cycle_life'].min():.0f}~{train_locked['cycle_life'].max():.0f}")
for name, spec in comparisons.items():
    seen = int(spec["pipe"].named_steps["scaler"].n_samples_seen_)
    print(name, spec["params"], "학습 셀", seen)
if len(train_locked) != 29 or int(loaded_again.named_steps["scaler"].n_samples_seen_) != 29:
    raise RuntimeError("학습 셀 수가 29가 아니다. Batch 3를 읽기 전에 멈춘다.")
if type(ridge_step).__name__ != "Ridge" or abs(float(ridge_step.alpha) - 1e-06) > 0:
    raise RuntimeError("잠긴 Ridge alpha가 1e-06이 아니다.")
ridge = loaded_again
print("평가 전 고정 확인: 모델 4개, 학습 셀 29. Batch 3는 아직 읽지 않음.")


파일 results/final_model.joblib
Ridge Ridge alpha=1e-06 max_iter=10000
Ridge 학습 셀 수 29
비교군은 P10 적합 유지, 재튜닝 없음. 학습 셀 29
train cycle_life 534~1054
DecisionTree max_depth=1, min_samples_leaf=3 학습 셀 29
XGBoost max_depth=2, n_estimators=50, learning_rate=0.05 학습 셀 29
LinearRegression(log10_var) 피처 log10_var 하나 학습 셀 29
평가 전 고정 확인: 모델 4개, 학습 셀 29. Batch 3는 아직 읽지 않음.


In [12]:
import numpy as np

from src.train import (
    append_batch3_performance,
    build_eval_predictions,
    load_features,
    mape_original,
)

# Batch 3는 이 셀에서만 읽는다. 예측도 이 셀에서 한 번. [D25] [D39]
batch3 = load_features(3)
print("읽는 파일 data/processed/features_batch3.csv", "행", len(batch3), "batch", sorted(batch3["batch"].unique()))
print("설계서 2장 44셀", "일치" if len(batch3) == 44 else "불일치")
if len(batch3) != 44:
    raise RuntimeError(f"Batch 3 셀 수가 44가 아니다: {len(batch3)}")

pred3 = build_eval_predictions(batch3, train_locked, ridge, comparisons, batch=3)
aligned3 = batch3.set_index("cell_id").loc[pred3["cell_id"]]
test3_mape = mape_original(aligned3["log_life"], np.log10(pred3["Ridge_예측"].to_numpy()))
print(f"Test MAPE 원값 {test3_mape:.6f}")

report3 = append_batch3_performance(test3_mape, n_cells=len(pred3))
print(report3.to_string(index=False))
print("저장", RESULTS_DIR / "model_performance.csv", "행", len(report3))

pred3_path = RESULTS_DIR / "predictions_batch3.csv"
pred3.to_csv(pred3_path, index=False)
print("저장", pred3_path, "행", len(pred3))

show3 = pred3.copy()
for column in ("cycle_life", "Ridge_예측", "DecisionTree_예측", "XGBoost_예측", "log10_var_단일_예측"):
    show3[column] = pred3[column].map(lambda value: f"{value:.3f}")
show3["오차%"] = pred3["오차%"].map(lambda value: f"{value:.3f}")
show3["|오차%|"] = pred3["|오차%|"].map(lambda value: f"{value:.3f}")
show3["log10_var"] = pred3["log10_var"].map(lambda value: f"{value:.3f}")
show3["qd_max_minus_2"] = pred3["qd_max_minus_2"].map(lambda value: f"{value:.6f}")
print(show3.to_string(index=False))


읽는 파일 data/processed/features_batch3.csv 행 44 batch [np.int64(3)]
설계서 2장 44셀 일치
Test MAPE 원값 24.945153
                        구분  MAPE (%)                                  비고
        Train (Batch 1 CV)     6.657 29 / 7 / 39. GroupKFold 평균. 학습 셀 29
  Valid (Batch 1 Hold-out)     4.888           29 / 7 / 39. hold-out 셀 7
            Test (Batch 2)    20.259               29 / 7 / 39. 테스트 셀 39
         Gap (Train-Valid)    -1.769           29 / 7 / 39. (+) : 과적합 의심
          Gap (Valid-Test)    15.371    29 / 7 / 39. (+) : 배치간 일반화 저하 의심
         Gap (Target-Test)    11.159      29 / 7 / 39. Target : 원논문 9.1%
            Test (Batch 3)    24.945                            테스트 셀 44
       Gap (Batch2-Batch3)     4.686                         테스트 배치 간 비교
Gap (Target-Test, Batch 3)    15.845                   Target : 원논문 9.1%
저장 results/model_performance.csv 행 9
저장 results/predictions_batch3.csv 행 44
cell_id             policy_readable cycle_life     수명_구간 log10_var qd_max_minus_2 log10_var

In [13]:
import pickle

from src.config import DATA_PROCESSED_DIR
from src.features import MODEL_FEATURES, delta_q
from src.train import LIFE_BANDS, LIFE_LONG, feature_range_table, life_band_table

pred2_saved = pd.read_csv(RESULTS_DIR / "predictions_batch2.csv")

def _show_bands(frame, title):
    bands = life_band_table(frame)
    view = bands.copy()
    for column in view.columns:
        if column in ("수명_구간", "셀_수", "과대예측_셀_수"):
            continue
        view[column] = bands[column].map(lambda value: f"{value:.3f}" if value == value else "")
    print(title)
    print(view.to_string(index=False))
    return bands

bands2 = _show_bands(pred2_saved, "Batch 2 수명 구간 (저장된 예측)")
bands3 = _show_bands(pred3, "Batch 3 수명 구간")
long_n = int((pred3["수명_구간"] == ">1,074").sum())
print("설계서 8장 Batch 3의 1,074 초과 16셀", "일치" if long_n == 16 else "불일치", "재현", long_n)

train_max = float(train_locked["cycle_life"].max())
print(
    "Batch 3 실제 최장", f"{pred3['cycle_life'].max():.0f}",
    "DecisionTree 예측 최대", f"{pred3['DecisionTree_예측'].max():.3f}",
    "XGBoost 예측 최대", f"{pred3['XGBoost_예측'].max():.3f}",
    "Ridge 예측 최대", f"{pred3['Ridge_예측'].max():.3f}",
)
print(
    "train 수명 최대", f"{train_max:.0f}",
    "DecisionTree가 train 최대보다 큰 셀", int((pred3["DecisionTree_예측"] > train_max).sum()),
    "XGBoost가 train 최대보다 큰 셀", int((pred3["XGBoost_예측"] > train_max).sum()),
    "DecisionTree가 1,074보다 큰 셀", int((pred3["DecisionTree_예측"] > LIFE_LONG).sum()),
    "XGBoost가 1,074보다 큰 셀", int((pred3["XGBoost_예측"] > LIFE_LONG).sum()),
)

scaler = ridge.named_steps["scaler"]
linear = ridge.named_steps["model"]
coef = np.asarray(linear.coef_, dtype=float).ravel()
intercept = float(np.asarray(linear.intercept_).reshape(-1)[0])
z_train = scaler.transform(train_locked.loc[:, list(MODEL_FEATURES)])
c_train = z_train * coef
print(
    "train 29 기여 평균",
    f"log10_var {c_train[:, 0].mean():.6f}",
    f"qd_max_minus_2 {c_train[:, 1].mean():.6f}",
)

def contribution_table(frame):
    z = scaler.transform(frame.loc[:, list(MODEL_FEATURES)])
    contrib = z * coef
    recon = intercept + contrib.sum(axis=1)
    gap = float(np.max(np.abs(recon - np.log10(frame["Ridge_예측"].to_numpy(float)))))
    rows = []
    for band in LIFE_BANDS:
        mask = (frame["수명_구간"] == band).to_numpy()
        if int(mask.sum()) == 0:
            continue
        part = frame.loc[mask]
        single = (part["log10_var_단일_예측"] - part["cycle_life"]) / part["cycle_life"] * 100.0
        rows.append({
            "수명_구간": band,
            "셀_수": int(mask.sum()),
            "log10_var_기여": float(contrib[mask, 0].mean()),
            "qd_기여": float(contrib[mask, 1].mean()),
            "Ridge_평균_부호_오차%": float(part["오차%"].mean()),
            "단일_평균_부호_오차%": float(single.mean()),
        })
    return pd.DataFrame(rows), gap

for label, frame in (("Batch 2", pred2_saved), ("Batch 3", pred3)):
    table, gap = contribution_table(frame)
    view = table.copy()
    for column in view.columns:
        if column in ("수명_구간", "셀_수"):
            continue
        view[column] = table[column].map(lambda value: f"{value:.3f}")
    print(label, "기여 재구성 최대 차이", f"{gap:.3e}")
    print(view.to_string(index=False))

def _show_range(frame, title):
    ranges = feature_range_table(frame)
    view = ranges.copy()
    view["Ridge_MAPE"] = ranges["Ridge_MAPE"].map(lambda value: f"{value:.3f}" if value == value else "")
    outside_mask = (frame["log10_var_범위"] != "안") | (frame["qd_max_minus_2_범위"] != "안")
    outside = int(outside_mask.sum())
    outside_mape = float(frame.loc[outside_mask, "|오차%|"].mean()) if outside else float("nan")
    print(title)
    print(view.to_string(index=False))
    print("두 피처 중 하나라도 범위 밖", outside, "Ridge MAPE", f"{outside_mape:.3f}")

_show_range(pred2_saved, "Batch 2 피처 train 범위 (저장된 예측)")
_show_range(pred3, "Batch 3 피처 train 범위")

# 3.0V ΔQ. 예측은 하지 않는다. [D08] [D30]
with (DATA_PROCESSED_DIR / "batch3.pkl").open("rb") as handle:
    payload = pickle.load(handle)
cells = payload["cells"]
cells = cells.loc[cells["use"]].reset_index(drop=True)
vdlin = np.asarray(payload["vdlin"], dtype=float)
voltage_index = int(np.argmin(np.abs(vdlin - 3.0)))
band_curves = []
dq_rows = []
for record in cells.itertuples(index=False):
    curve = delta_q(payload["qdlin"][record.cell_id])
    point = float(curve[voltage_index])
    dq_rows.append({"cell_id": record.cell_id, "dq_3v": point})
    if 700 <= float(record.cycle_life) <= 1100:
        band_curves.append(curve)
median_curve = np.median(np.vstack(band_curves), axis=0)
repro = float(median_curve[voltage_index])
rounded = float(f"{repro:.3f}")
print("3.0V 인덱스", voltage_index, "전압", f"{vdlin[voltage_index]:.4f}", "셀 수", len(band_curves))
print("설계서", "-0.024", "재현", f"{repro:.4f}", "소수 셋째", f"{rounded:.3f}", "일치" if rounded == -0.024 else "불일치")
pred3 = pred3.merge(pd.DataFrame(dq_rows), on="cell_id", how="left")
if pred3["dq_3v"].isna().any():
    raise RuntimeError("Batch 3 예측 셀에 3.0V ΔQ가 없다.")

def _is_new(policy):
    return str(policy).endswith("newstructure")

train_new = int(train_locked["policy_readable"].map(_is_new).sum())
print("Batch 1 train newstructure", train_new, "설계 0셀", "일치" if train_new == 0 else "불일치")

def newstructure_table(frame, batch_name):
    rows = []
    flag = frame["policy_readable"].map(_is_new)
    for label, part in (("newstructure", frame.loc[flag]), ("일반", frame.loc[~flag])):
        rows.append({
            "배치": batch_name,
            "구분": label,
            "셀_수": int(len(part)),
            "Ridge_MAPE": float(part["|오차%|"].mean()) if len(part) else float("nan"),
            "평균_부호_오차%": float(part["오차%"].mean()) if len(part) else float("nan"),
        })
    return pd.DataFrame(rows)

ns = pd.concat([
    newstructure_table(pred2_saved, "Batch 2"),
    newstructure_table(pred3, "Batch 3"),
], ignore_index=True)
ns_view = ns.copy()
for column in ("Ridge_MAPE", "평균_부호_오차%"):
    ns_view[column] = ns[column].map(lambda value: f"{value:.3f}" if value == value else "")
print(ns_view.to_string(index=False))

kept = {"b2c7", "b2c8", "b2c9", "b2c15", "b2c16"}
rank2 = pred2_saved["|오차%|"].rank(ascending=False, method="min")
kept_rows = pred2_saved.loc[pred2_saved["cell_id"].isin(kept)].copy()
kept_rows["오차_순위"] = rank2.loc[kept_rows.index].astype(int)
kept_view = pd.DataFrame({
    "cell_id": kept_rows["cell_id"],
    "오차%": kept_rows["오차%"].map(lambda value: f"{value:.3f}"),
    "|오차%|": kept_rows["|오차%|"].map(lambda value: f"{value:.3f}"),
    "오차_순위": kept_rows["오차_순위"],
    "상위5": kept_rows["오차_순위"] <= 5,
})
print("D05 테스트 유지 셀, Batch 2 저장 예측")
print(kept_view.sort_values("오차_순위").to_string(index=False))

top3 = pred3.nlargest(5, "|오차%|").copy()
train_policies = set(train_locked["policy_readable"])
top_view = pd.DataFrame({
    "cell_id": top3["cell_id"],
    "정책": top3["policy_readable"],
    "실제_수명": top3["cycle_life"].map(lambda value: f"{value:.0f}"),
    "예측_수명": top3["Ridge_예측"].map(lambda value: f"{value:.3f}"),
    "오차%": top3["오차%"].map(lambda value: f"{value:.3f}"),
    "수명_구간": top3["수명_구간"],
    "log10_var_범위": top3["log10_var_범위"],
    "qd_범위": top3["qd_max_minus_2_범위"],
    "dq_3v": top3["dq_3v"].map(lambda value: f"{value:.4f}"),
    "정책_train에_있음": top3["policy_readable"].isin(train_policies),
    "D05_테스트_유지": top3["cell_id"].isin(kept),
})
print("Batch 3 오차 상위 5")
print(top_view.to_string(index=False))

short2 = pred2_saved.loc[pred2_saved["수명_구간"] == "<534"]
long3 = pred3.loc[pred3["수명_구간"] == ">1,074"]
compare_rows = []
for name, part, direction in (
    ("Batch 2", short2, "짧은 쪽"),
    ("Batch 3", long3, "긴 쪽"),
):
    compare_rows.append({
        "배치": name,
        "외삽_방향": direction,
        "셀_수": int(len(part)),
        "Ridge_MAPE": float(part["|오차%|"].mean()),
        "트리_MAPE": float(np.mean(np.abs(part["DecisionTree_예측"] - part["cycle_life"]) / part["cycle_life"]) * 100.0),
        "평균_부호_오차%": float(part["오차%"].mean()),
    })
compare = pd.DataFrame(compare_rows)
compare_view = compare.copy()
for column in ("Ridge_MAPE", "트리_MAPE", "평균_부호_오차%"):
    compare_view[column] = compare[column].map(lambda value: f"{value:.3f}")
print("외삽 방향 비교")
print(compare_view.to_string(index=False))


Batch 2 수명 구간 (저장된 예측)
    수명_구간  셀_수 Ridge_MAPE Ridge_평균_부호_오차%  과대예측_셀_수 DecisionTree MAPE XGBoost MAPE log10_var 단일 MAPE
     <534   30     15.121           1.124        15            48.096       48.483            33.865
534~1,074    7     37.701          31.999         6             9.328        9.349            18.023
   >1,074    2     36.277          36.277         2            24.984       18.493             6.060
Batch 3 수명 구간
    수명_구간  셀_수 Ridge_MAPE Ridge_평균_부호_오차%  과대예측_셀_수 DecisionTree MAPE XGBoost MAPE log10_var 단일 MAPE
     <534    0                                    0                                                 
534~1,074   28     25.573          25.178        27             9.540        9.849             8.981
   >1,074   16     23.847          17.812        12            34.099       29.122            17.881
설계서 8장 Batch 3의 1,074 초과 16셀 일치 재현 16
Batch 3 실제 최장 1935 DecisionTree 예측 최대 872.099 XGBoost 예측 최대 947.558 Ridge 예측 최대 2266.694
train 수명 최대 1054 DecisionTre

## Gap(Batch2-Batch3)

1. 목적: Batch 3가 Batch 2보다 얼마나 나쁜지, 피처가 특정 배치에만 맞았는지를 본다. [D25] [D39]
2. 결과: Test(Batch 3) 24.945, Test(Batch 2) 20.259, Gap(Batch2-Batch3) = 4.686. Gap(Target-Test, Batch 3) = 15.845. Batch 3 평균 부호 오차는 +22.500%이고 과대예측은 39/44다. 1,074 초과는 16셀로 설계서와 같다. DecisionTree 예측 최대는 872.099, XGBoost는 947.558이라 train 최대 1,054를 넘는 셀은 0이다.
3. 해석: 4.686%p는 Gap(Valid-Test) 15.371보다 작다. Batch 1 안의 Gap(Train-Valid)는 −1.769라, 학습 배치 내부의 과적합은 아니다. 두 테스트가 같이 나쁘고 Batch 3가 조금 더 나쁘다. Batch 2 평균은 수명 534 미만 30셀(MAPE 15.121, 평균 부호 +1.124)이 끌어내린 값이다. Batch 3에는 그 짧은 셀이 0개라 상쇄가 없다. newstructure만 보면 Batch 2의 9셀 MAPE 37.384가 Batch 3 전체 24.945보다 크다. 그래서 접미사 자체가 Batch 3를 더 어렵게 만든 것은 아니다. 공통 원인은 Batch 1의 ΔQ 스케일이다. Batch 3에서 log10_var가 train보다 아래인 셀은 24개이고 MAPE 28.457이다. 기여 평균은 534~1,074에서 분산 +0.111, qd_max_minus_2 +0.039, 1,074 초과에서 +0.268, +0.040이다. 둘 다 예측 수명을 늘린다. 1,074 초과의 log10_var 단일 모델 평균 부호는 −11.909%라, 긴 쪽 과대예측을 키우는 쪽은 qd_max_minus_2다. DAY 1 Q3는 충전 곡선이 잡히기 시작한 시점이 배치마다 달라 같은 전압 인덱스에 다른 구간이 들어온다고 적었다. 3.0V 중앙 ΔQ −0.024 재현은 그 오프셋과 맞다. 원고에 3.25V 근처가 튄다는 문장은 없어 그 점과는 연결하지 않았다.
4. 시사점: 피처가 Batch 1의 ΔQ 깊이에 묶여 있다. 배치 번호는 넣지 않고, 이 결과로 모델·피처·분할을 바꾸지 않는다. [D30]

## 외삽 방향

| 배치 | 외삽 방향 | 셀 수 | Ridge MAPE | 트리 MAPE | 평균 부호 % |
| --- | --- | --- | --- | --- | --- |
| Batch 2 | 짧은 쪽 | 30 | 15.121 | 48.096 | +1.124 |
| Batch 3 | 긴 쪽 | 16 | 23.847 | 34.099 | +17.812 |

짧은 쪽은 Ridge가 수명을 534 아래로 내리지만 부호가 상쇄된다. 긴 쪽은 Ridge 평균 부호가 +17.812%로 과대예측이고, 트리는 872.099에서 멈춰 MAPE가 더 크다. 최장 b3c38(1,935)만은 Ridge 1,495.521, 오차 −22.712%로 과소예측이다. [D27] [D29]

## 설계 예상 vs 결과

| 설계 | 결과 | 판정 |
| --- | --- | --- |
| Batch 3는 Batch 2 다음의 선택 검증 | 잠긴 Ridge로 한 번. Test 24.945 | 일치 |
| 사용 셀 44 | 44 | 일치 |
| 1,074보다 긴 셀 16개 | 16 | 일치 |
| 수명 700~1,100의 3.0V 중앙 ΔQ −0.024Ah | −0.0241, 소수 셋째 −0.024, n=28 | 일치 |
| newstructure는 Batch 3 전부, Batch 1 train은 0 | 44와 0 | 일치 |
| 트리는 학습 범위 위를 예측하지 못함 | DecisionTree 최대 872.099, train 최대 초과 0셀 | 일치 |
| ΔQ가 얕으면 Batch 3 수명을 과대예측 | 평균 부호 +22.500, 과대예측 39/44 | 부분 적중. 최장 b3c38은 −22.712 |

## Batch 3 오차 상위 5셀

| cell_id | 정책 | 실제 | Ridge 예측 | 오차% | 수명 구간 | log10_var | qd_max_minus_2 | 3.0V ΔQ | train 정책 | D05 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| b3c2 | 5.6C(19%)-4.6C-newstructure | 1,267 | 2266.694 | +78.902 | >1,074 | 아래 | 아래 | +0.0035 | 없음 | 아님 |
| b3c40 | 5.6C(19%)-4.6C-newstructure | 796 | 1265.751 | +59.014 | 534~1,074 | 아래 | 안 | −0.0186 | 없음 | 아님 |
| b3c41 | 5.6C(36%)-4.3C-newstructure | 786 | 1190.157 | +51.419 | 534~1,074 | 아래 | 안 | −0.0180 | 없음 | 아님 |
| b3c5 | 5.6C(36%)-4.3C-newstructure | 828 | 1201.736 | +45.137 | 534~1,074 | 안 | 아래 | −0.0243 | 없음 | 아님 |
| b3c36 | 5.6C(36%)-4.3C-newstructure | 923 | 1329.525 | +44.044 | 534~1,074 | 아래 | 아래 | −0.0218 | 없음 | 아님 |

다섯 셀 모두 newstructure이고 정책은 Batch 1 train에 없다. D05로 테스트에 남긴 b2c7, b2c8, b2c9, b2c15, b2c16은 이 표에 없다. 저장된 Batch 2 예측에서 그 다섯 중 오차 상위 5에 들어가는 셀은 b2c9(순위 2, +62.690%)뿐이다.

## Batch 3 요약

Test MAPE 24.945, Gap(Batch2-Batch3) 4.686, Gap(Target-Test, Batch 3) 15.845. 셀 44개, 1,074 초과 16개.
평균 부호 +22.500. 534 미만이 없어 Batch 2처럼 짧은 쪽 상쇄가 없다. 3.0V 중앙 ΔQ는 −0.024로 설계와 같다.
트리는 예측 최대 872.099에서 멈춘다. Ridge는 긴 쪽 평균 부호 +17.812이지만 최장 b3c38은 −22.712다.
모델은 유지한다. 원인은 Batch 1의 ΔQ 스케일과 충전 곡선 시작 시점 차이고, 배치 번호는 넣지 않는다.
